# Select the moving-average window and consecutive alarms

This follows the TEP alarm-selection procedure. Detection scores use only the three endogenous process variables; all six variables remain available as model predictors. Autoregressive models are trained on healthy training runs in notebook 1.0. For each window, the threshold is the maximum smoothed score in the pre-fault part of the faulty validation trajectories. Each faulty trajectory is split into its healthy first phase and faulty second phase; window/consecutive-alarm settings are selected using these two phases. The held-out test split is not used for tuning.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: <repo>


In [2]:
import numpy as np
import pandas as pd

from carl.causalai_process import ENDOGENOUS_NAMES


data_dir = PROJECT_ROOT / "data" / "interim" / "causalAIProcess"
graph_file = PROJECT_ROOT / "data" / "processed" / "causalAIProcess" / "causal_graph.csv"
models_dir = PROJECT_ROOT / "models" / "causalAIProcess" / "max_lag_selection"
results_dir = PROJECT_ROOT / "reports" / "results" / "causalAIProcess"
results_dir.mkdir(parents=True, exist_ok=True)

training_file_name = "traces_scaled.parquet"
MIN_WINDOW = 1
MAX_WINDOW = 8
MIN_CONSECUTIVE_ALARMS = 5
MAX_CONSECUTIVE_ALARMS = 8

In [3]:
df = pd.read_parquet(data_dir / training_file_name)
runs = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "causalAIProcess" / "runs.csv")
causal_graph = pd.read_csv(graph_file)
selected_lag_path = models_dir / "selected_max_lag.csv"
if not selected_lag_path.is_file():
    raise FileNotFoundError("Run 1.0-max_lag_selection.ipynb before selecting alarms.")

max_lag = int(pd.read_csv(selected_lag_path).loc[0, "best_max_lag"])
feature_names = [
    column
    for column in df.columns
    if column not in ["run_id", "time_step", "time_seconds", "anomaly", "split"]
]
child_parents_pairs = {
    child: sorted(causal_graph.loc[causal_graph["target"] == child, "source"].unique())
    for child in feature_names
}
model_features = {
    child: list(dict.fromkeys(parents + [child]))
    for child, parents in child_parents_pairs.items()
}
process_children = [name for name in feature_names if name in ENDOGENOUS_NAMES]
if set(process_children) != set(ENDOGENOUS_NAMES):
    raise ValueError("Traces are missing endogenous process variables.")

if df.groupby("run_id").size().nunique() != 1:
    raise ValueError("All trajectories must have equal length for alarm evaluation.")
n_samples_per_ts = int(df.groupby("run_id").size().iloc[0])
if max_lag >= n_samples_per_ts:
    raise ValueError("Selected maximum lag is longer than an input trajectory.")

healthy_training_ids = runs.loc[
    runs["split"].eq("train") & runs["fault_type"].eq("healthy"), "run_id"
].tolist()
validation_ids = runs.loc[runs["split"].eq("validation"), "run_id"].tolist()
validation_runs = runs.loc[runs["run_id"].isin(validation_ids)].set_index("run_id")
df_faultfree_train = df.loc[df["run_id"].isin(healthy_training_ids)]
df_validation = df.loc[df["run_id"].isin(validation_ids)]
n_faultfree_ts = len(healthy_training_ids)
n_validation_ts = len(validation_ids)
n_process_children = len(process_children)

print(f"Selected max lag: {max_lag}")
print(f"Healthy training runs: {n_faultfree_ts}; validation runs: {n_validation_ts}")
display(validation_runs.groupby("fault_type").size())

Selected max lag: 2
Healthy training runs: 350; validation runs: 300


fault_type
healthy                     75
hydraulic_coupling_shift    75
reaction_coupling_shift     75
thermal_coupling_shift      75
dtype: int64

In [4]:
def create_lagged_data(dataframe, child, lag):
    features = dataframe[model_features[child]]
    lagged = pd.concat(
        [features.shift(step).add_suffix(f"_lag{step}") for step in range(1, lag + 1)],
        axis=1,
    ).iloc[lag:]
    x = np.ones((len(lagged), lagged.shape[1] + 1))
    x[:, 1:] = lagged.to_numpy()
    y = dataframe[child].iloc[lag:].to_numpy()
    return x, y


residuals_by_child = []
for child in process_children:
    beta_path = models_dir / f"beta_child_{child}_lag_{max_lag}.npy"
    if not beta_path.is_file():
        raise FileNotFoundError(f"Missing fitted model weights: {beta_path}")
    beta = np.load(beta_path)
    child_residuals = []
    for _, trajectory in df_faultfree_train.groupby("run_id", sort=False):
        x_train, y_train = create_lagged_data(trajectory, child, max_lag)
        child_residuals.append(y_train - x_train @ beta)
    residuals_by_child.append(np.concatenate(child_residuals))

res_train = np.column_stack(residuals_by_child)
sample_cov = (res_train.T @ res_train) / len(res_train)
sample_precision = np.linalg.pinv(sample_cov)
np.save(models_dir / "sample_precision_train.npy", sample_precision)
print(f"Training residual matrix: {res_train.shape}")

Training residual matrix: (349300, 3)


In [5]:
scores_by_run = {}
labels_by_run = {}
time_steps_by_run = {}

for run_id, trajectory in df_validation.groupby("run_id", sort=False):
    trajectory = trajectory.sort_values("time_step")
    residuals = np.empty((n_samples_per_ts - max_lag, n_process_children))
    for child_idx, child in enumerate(process_children):
        beta = np.load(models_dir / f"beta_child_{child}_lag_{max_lag}.npy")
        x, y = create_lagged_data(trajectory, child, max_lag)
        residuals[:, child_idx] = y - x @ beta

    scores_by_run[run_id] = np.mean((residuals @ sample_precision) * residuals, axis=1)
    labels_by_run[run_id] = trajectory["anomaly"].to_numpy()[max_lag:]
    time_steps_by_run[run_id] = trajectory["time_step"].to_numpy()[max_lag:]

faulty_validation_ids = validation_runs.index[
    validation_runs["fault_type"] != "healthy"
].tolist()
if not faulty_validation_ids:
    raise ValueError("Validation split must include faulty trajectories.")

windows = range(MIN_WINDOW, MAX_WINDOW + 1)
consecutive_alarms_list = range(
    MIN_CONSECUTIVE_ALARMS, MAX_CONSECUTIVE_ALARMS + 1
)
smoothed_scores = {}
thresholds_by_window = {}
for window in windows:
    smoothed_scores[window] = {}
    for run_id, scores in scores_by_run.items():
        smoothed_scores[window][run_id] = (
            pd.Series(scores).rolling(window=window, min_periods=window).mean().to_numpy()
        )
    healthy_phase_scores = []
    for run_id in faulty_validation_ids:
        onset = int(validation_runs.loc[run_id, "fault_start_step"])
        aligned_steps = time_steps_by_run[run_id][MAX_WINDOW - 1 :]
        aligned_scores = smoothed_scores[window][run_id][MAX_WINDOW - 1 :]
        healthy_phase_scores.append(aligned_scores[aligned_steps < onset])
    healthy_scores = np.concatenate(healthy_phase_scores)
    healthy_scores = healthy_scores[np.isfinite(healthy_scores)]
    if healthy_scores.size == 0:
        raise ValueError(f"No finite healthy validation scores for window {window}.")
    thresholds_by_window[window] = np.max(healthy_scores)

thresholds = pd.DataFrame(
    [{"Window Size": window, "Threshold": threshold}
     for window, threshold in thresholds_by_window.items()]
)
thresholds.to_csv(results_dir / "alarm_thresholds_by_window.csv", index=False)
display(thresholds)

,Window Size,Threshold
0,1,9.661041
1,2,6.474015
2,3,5.394820
3,4,4.607949
4,5,4.072094
5,6,3.420817
6,7,3.223454
7,8,2.975990


In [6]:
def binary_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    tpr = tp / (tp + fn) if tp + fn else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    precision = tp / (tp + fp) if tp + fp else 0.0
    f1 = 2 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    return accuracy, tpr, fpr, f1


df_results = []
for window in windows:
    threshold = thresholds_by_window[window]
    raw_breaches = {
        run_id: (smoothed_scores[window][run_id][MAX_WINDOW - 1 :] > threshold).astype(int)
        for run_id in faulty_validation_ids
    }

    for consecutive_alarms in consecutive_alarms_list:
        predictions_by_run = {}
        eval_masks_by_run = {}
        for run_id, breaches in raw_breaches.items():
            if consecutive_alarms > 1:
                predictions = (
                    pd.Series(breaches)
                    .rolling(window=consecutive_alarms, min_periods=consecutive_alarms)
                    .sum()
                    .eq(consecutive_alarms)
                    .fillna(False)
                    .to_numpy(dtype=int)
                )
            else:
                predictions = breaches
            predictions_by_run[run_id] = predictions
            eval_mask = np.ones(len(predictions), dtype=bool)
            eval_mask[: consecutive_alarms - 1] = False
            eval_masks_by_run[run_id] = eval_mask

        y_true = np.concatenate(
            [
                labels_by_run[run_id][MAX_WINDOW - 1 :][eval_masks_by_run[run_id]]
                for run_id in faulty_validation_ids
            ]
        )
        y_pred = np.concatenate(
            [predictions_by_run[run_id][eval_masks_by_run[run_id]] for run_id in faulty_validation_ids]
        )
        sample_accuracy, sample_tpr, sample_fpr, sample_f1 = binary_metrics(y_true, y_pred)

        phase_true = []
        phase_pred = []
        detection_delays = []
        for run_id in faulty_validation_ids:
            run = validation_runs.loc[run_id]
            steps = time_steps_by_run[run_id][MAX_WINDOW - 1 :]
            predictions = predictions_by_run[run_id]
            onset = int(run["fault_start_step"])
            before_fault = steps < onset
            during_fault = steps >= onset
            phase_true.extend([0, 1])
            phase_pred.extend([
                int(predictions[before_fault].any()),
                int(predictions[during_fault].any()),
            ])
            fault_indices = np.flatnonzero(during_fault & (predictions == 1))
            if len(fault_indices):
                detection_delays.append(int(steps[fault_indices[0]] - onset))

        phase_accuracy, phase_tpr, phase_fpr, phase_f1 = binary_metrics(
            np.asarray(phase_true), np.asarray(phase_pred)
        )
        df_results.append(
            {
                "Window Size": window,
                "Consecutive Alarms": consecutive_alarms,
                "Threshold": threshold,
                "Sample Accuracy": sample_accuracy,
                "Sample TPR": sample_tpr,
                "Sample FPR": sample_fpr,
                "Sample F1": sample_f1,
                "Trajectory-Phase Accuracy": phase_accuracy,
                "Trajectory-Phase TPR": phase_tpr,
                "Trajectory-Phase FPR": phase_fpr,
                "Trajectory-Phase F1": phase_f1,
                "Median Anomaly Detection Delay": (
                    float(np.median(detection_delays)) if detection_delays else np.nan
                ),
            }
        )

df_results = pd.DataFrame(df_results)
results_path = results_dir / "window_and_consecutive_alarm_selection.csv"
df_results.to_csv(results_path, index=False)
f1_col = "Trajectory-Phase F1"
delay_col = "Median Anomaly Detection Delay"

best_f1 = df_results[f1_col].max()
best_candidates = df_results.loc[df_results[f1_col].eq(best_f1)]
best_candidates = best_candidates.dropna(subset=[delay_col])
if best_candidates.empty:
    raise ValueError("No best-F1 configuration has a valid detection delay.")
best_idx = best_candidates[delay_col].idxmin()

best_window = int(df_results.loc[best_idx, "Window Size"])
best_cons_alarms = int(df_results.loc[best_idx, "Consecutive Alarms"])
best_threshold = float(df_results.loc[best_idx, "Threshold"])

print("Highest trajectory-phase F1:")
print(f"\tWindow Size = {best_window}")
print(f"\tConsecutive Alarms = {best_cons_alarms}")
print(f"\tThreshold = {best_threshold:.6g}")
print(f"Saved results to {results_path}")
display(df_results.sort_values("Trajectory-Phase F1", ascending=False).head(20))

Highest trajectory-phase F1:
	Window Size = 8
	Consecutive Alarms = 5
	Threshold = 2.97599
Saved results to <repo>\reports\results\causalAIProcess\window_and_consecutive_alarm_selection.csv


,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
28,8,5,2.975990,0.513795,0.400144,0.0,0.571576,0.904444,0.808889,0.0,0.894349,31.0
29,8,6,2.975990,0.508425,0.394133,0.0,0.565417,0.897778,0.795556,0.0,0.886139,30.0
30,8,7,2.975990,0.503350,0.388500,0.0,0.559597,0.897778,0.795556,0.0,0.886139,33.0
20,6,5,3.420817,0.492009,0.373267,0.0,0.543619,0.895556,0.791111,0.0,0.883375,32.0
24,7,5,3.223454,0.501853,0.385411,0.0,0.556385,0.895556,0.791111,0.0,0.883375,30.5
25,7,6,3.223454,0.496398,0.379311,0.0,0.550001,0.893333,0.786667,0.0,0.880597,33.0
31,8,8,2.975990,0.498473,0.383122,0.0,0.553996,0.893333,0.786667,0.0,0.880597,34.0
26,7,7,3.223454,0.491204,0.373544,0.0,0.543913,0.888889,0.777778,0.0,0.875000,34.0
21,6,6,3.420817,0.486396,0.366983,0.0,0.536924,0.886667,0.773333,0.0,0.872180,33.0
22,6,7,3.420817,0.481042,0.361033,0.0,0.530528,0.862222,0.724444,0.0,0.840206,33.0
